# Tutorial - Ramsey

**Ramsey interferometry** measures the free-induction decay of a spin superposition:
$$\text{laser} \to \tfrac{\pi}{2} \to \text{free}(\tau) \to \tfrac{\pi}{2} \to \text{readout}.$$
The signal oscillates at the detuning and decays on $T_2^*$. We fit:
- **no-HF**: an exponential decay (`expo`);
- **HF (Doherty/Duarte)**: a damped cosine (`damp_cos`) whose frequency gives the hyperfine/Zeeman splitting $B_0=\omega/2\pi\mu_e$.

We also take the **FFT** of the fringe and run a **spin echo** (Hahn echo) sequence, which refocuses static dephasing with a central $\pi$ pulse and decays on the longer $T_2$.

`RECOMPUTE = True` runs the full sweep; `False` loads the shipped reduced cache.

In [ ]:
RECOMPUTE = True
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from examples import protocol_runs as engine
import core.plot_funcs as pf

## 1. Ramsey fringes + fits

In [ ]:
ramsey = engine.run_or_load(
    "ramsey",
    lambda: engine.compute_ramsey(["no-HF", "Doherty", "Duarte"], np.linspace(0.05, 10.0, 200)),
    recompute=RECOMPUTE,
)
_, fits = engine.plot_ramsey_result(ramsey)
for model, fit in fits.items():
    print(model, {k: round(v[0], 3) for k, v in fit.items()})
plt.show()

## 2. FFT of the Ramsey signal

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for model, entry in ramsey["signals"].items():
    pf.plot_fft(ramsey["free_times"], np.real(entry["signal"]), ax=ax, label=model)
plt.show()

## 3. Spin echo (Hahn echo)

In [ ]:
echo = engine.run_or_load(
    "spin_echo",
    lambda: engine.compute_spin_echo(["no-HF"], np.linspace(0.05, 5.0, 40)),
    recompute=RECOMPUTE,
)
engine.plot_spin_echo_result(echo)
plt.show()

## 4. Ramsey vs magnetic field and Rabi frequency

The masters Ramsey scan sweeps $\Omega_r \in \{\pi, 2\pi, 10\pi\}/T_2$ and $B \in \{0, 100, 510, 1020\}$ G (a fringe per combination). **Long calculation** (a full $\tau$-sweep per combination) — set `RECOMPUTE=True` to run.

In [ ]:
T2_GS = 1.5  # us
for om_tag, om_r in [("piT2", np.pi / T2_GS), ("2piT2", 2 * np.pi / T2_GS),
                     ("10piT2", 10 * np.pi / T2_GS)]:
    for B in [0.0, 100.0, 510.0, 1020.0]:
        data = engine.run_or_load(
            f"ramsey_{om_tag}_b{int(B)}",
            lambda b=B, o=om_r: engine.compute_ramsey(
                ["no-HF"], np.linspace(0.05, 10.0, 200), field=b, rabi=o),
            recompute=RECOMPUTE,
        )
        engine.plot_ramsey_result(data, save_as=f"ramsey_{om_tag}_b{int(B)}.png")
        plt.show()

In [ ]:
# --- Smoke test: tiny self-contained run to confirm this notebook works ---
try:
    _r = engine.compute_ramsey(["no-HF"], np.linspace(0.05, 3.0, 6))
    engine.plot_ramsey_result(_r, save_as=None)
    pf.plot_fft(_r["free_times"], np.real(_r["signals"]["no-HF"]["signal"]))
    engine.plot_spin_echo_result(
        engine.compute_spin_echo(["no-HF"], np.linspace(0.05, 3.0, 6)), save_as=None)
    engine.plot_ramsey_result(
        engine.compute_ramsey(["no-HF"], np.linspace(0.05, 3.0, 6),
                              field=100.0, rabi=np.pi / 1.5), save_as=None)
    # HF (14-level Doherty) Ramsey — tiny
    engine.plot_ramsey_result(
        engine.compute_ramsey(["Doherty"], np.linspace(0.05, 3.0, 6), field=100.0), save_as=None)
    plt.close("all")
    print("SMOKE OK: Ramsey / FFT / spin-echo / multi-B / HF")
except Exception as _e:
    print("SMOKE FAIL:", repr(_e))